In [3]:
import os
import cv2

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from config import WLASL_RAW_DATA
from src.utils import HandDetection, PoseDetection, FaceDetection

video_id = "49596"
input_video = os.path.join(WLASL_RAW_DATA, f"{video_id}.mp4")
output_video = os.path.join(r"D:\SignDetection", f"{video_id}_skeleton.mp4")


cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise RuntimeError("Cannot open video.")


hand_detection = HandDetection(
    min_hand_detection_confidence=0.3
)

pose_detection = PoseDetection(
    min_pose_detection_confidence=0.3
)

face_detection = FaceDetection()


fps = cap.get(cv2.CAP_PROP_FPS)

if fps <= 0:
    fps = 25


width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"width: {width}, height: {height}")
print(f"fps: {fps}")
print(f"Output: {output_video}")


# =========================
# Video Writer
# =========================

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (width, height)
)

if not writer.isOpened():
    raise RuntimeError("Cannot open video writer.")


frame_index = 0

while True:

    success, frame = cap.read()

    if not success:
        print("End of video.")
        break

    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    timestamp_ms = int(
        frame_index * 1000 / fps
    )


    # =========================
    # Hand detection
    # =========================

    detection_hand_results = hand_detection.detect_video(
        rgb_frame,
        timestamp_ms
    )


    # =========================
    # Pose detection
    # =========================

    detection_pose_results = pose_detection.detect_video(
        rgb_frame,
        timestamp_ms
    )


    # =========================
    # Face detection
    # =========================

    detection_face_results = face_detection.detect_video(
        rgb_frame,
        timestamp_ms
    )


    # =========================
    # Draw hand
    # =========================

    output = hand_detection.draw_landmarks_on_image(
        rgb_frame.copy(),
        detection_hand_results
    )


    # =========================
    # Draw pose
    # =========================

    output = pose_detection.draw_landmarks_on_image(
        output,
        detection_pose_results
    )


    # =========================
    # Draw lips
    # =========================

    output = face_detection.draw_lips_on_image(
        output,
        detection_face_results
    )


    # =========================
    # RGB -> BGR
    # =========================

    output = cv2.cvtColor(
        output,
        cv2.COLOR_RGB2BGR
    )


    # =========================
    # Save frame to video
    # =========================

    writer.write(output)


    # =========================
    # Display
    # =========================

    cv2.imshow(
        "Landmarks + Lips",
        output
    )


    frame_index += 1


    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# =========================
# Release
# =========================

cap.release()
writer.release()

cv2.destroyAllWindows()

hand_detection.close()

print("Done.")
print(f"Saved video to: {output_video}")

width: 736, height: 414
fps: 29.97
Output: D:\SignDetection\49596_skeleton.mp4
End of video.
Done.
Saved video to: D:\SignDetection\49596_skeleton.mp4


In [ ]:
import os
import cv2
import numpy as np

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from config import WLASL_RAW_DATA
from src.utils import HandDetection, PoseDetection

input_video = os.path.join(WLASL_RAW_DATA, "69544.mp4")
output_video = os.path.join(r"D:\SignDetection", "69544_skeleton.mp4")

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise RuntimeError("Cannot open video.")

hand_detection = HandDetection(min_hand_detection_confidence=0.4)
pose_detection = PoseDetection(min_pose_detection_confidence=0.4)

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 25

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video size: {width} x {height}")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

video_writer = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (width, height),
)

frame_index = 0

while True:
    success, frame = cap.read()

    if not success:
        print("End of video.")
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    timestamp_ms = int(frame_index * 1000 / fps)

    # Detect
    hand_results = hand_detection.detect_video(rgb_frame, timestamp_ms)
    pose_results = pose_detection.detect_video(rgb_frame, timestamp_ms)

    # ==============================
    # Tạo nền đen
    # ==============================
    canvas = np.zeros((height, width, 3), dtype=np.uint8)

    # Vẽ skeleton lên nền đen
    canvas = pose_detection.draw_landmarks_on_image(canvas, pose_results)
    canvas = hand_detection.draw_landmarks_on_image(canvas, hand_results)

    # Nếu hàm draw trả về RGB thì chuyển sang BGR để ghi video
    output = cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR)

    video_writer.write(output)

    cv2.imshow("Skeleton", output)

    frame_index += 1

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
video_writer.release()
cv2.destroyAllWindows()

hand_detection.close()

print(f"Saved to: {output_video}")

In [10]:
import os
import cv2
import numpy as np

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from config import WLASL_RAW_DATA
from src.utils import HandDetection, PoseDetection, FaceDetection

input_video = os.path.join(WLASL_RAW_DATA, "69544.mp4")
cap = cv2.VideoCapture(input_video)
face_detection = FaceDetection()

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 0:
    fps = 25

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video size: {width} x {height}")

frame_index = 0

while True:
    success, frame = cap.read()

    if not success:
        print("End of video.")
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    timestamp_ms = int(frame_index * 1000 / fps)

    # Detect
    results = face_detection.detect_video(rgb_frame, timestamp_ms)


    # Vẽ skeleton lên nền đen
    frame = face_detection.draw_lips_on_image(rgb_frame, results)

    cv2.imshow("Skeleton", frame)
    frame_index += 1

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()



Video size: 1280 x 720
End of video.
